# Notebook 7.1  A unified loader and a data audit for Arabic corpora

**Goal.** Ingest two Arabic speech corpora into one common format and report the statistics that drive a corpus choice (Chapter 7).

**What it does.** (1) defines a shared per-utterance schema; (2) loads two corpora into it, with a real Common Voice loader and a graceful fallback to small synthetic samples so the notebook runs offline; (3) reports total hours, speakers, and hours per dialect and per gender; (4) flags speaker overlap that would break a leak-free split; (5) computes Fleiss' kappa on a small multi-annotator dialect-labelling example.

No downloads are required. To use real data, set `CV_TSV` to a Common Voice Arabic `validated.tsv` and re-run.


## 1. A shared schema for an utterance

Every corpus stores its metadata differently. The first step is to map each one onto the same five fields: audio path, transcript, speaker, dialect, and duration (plus gender where known).


In [ ]:
from dataclasses import dataclass, asdict
import pandas as pd, numpy as np

@dataclass
class Utterance:
    audio_path: str
    transcript: str
    speaker: str
    dialect: str          # e.g. MSA, Gulf, Egyptian, Levantine, Maghrebi
    gender: str           # 'm', 'f', or 'unknown'
    duration_s: float
    corpus: str = ''

COLUMNS = ['audio_path','transcript','speaker','dialect','gender','duration_s','corpus']

def to_frame(utts):
    return pd.DataFrame([asdict(u) for u in utts], columns=COLUMNS)
print('schema fields:', COLUMNS)


## 2. Loading two corpora into the shared schema

`load_commonvoice` shows the real pattern: read the TSV, rename columns, and attach durations. 
If the file is absent, `make_synthetic` builds two small corpora so the rest of the notebook runs. 
Each loader returns rows in the shared schema, and we concatenate them into one table with a `corpus` column.


In [ ]:
CV_TSV = 'validated.tsv'   # point this at a real Common Voice Arabic validated.tsv to use real data

def load_commonvoice(tsv_path, clip_durations='clip_durations.tsv'):
    df = pd.read_csv(tsv_path, sep='\t')
    dur = {}
    try:
        d = pd.read_csv(clip_durations, sep='\t')           # path, duration[ms]
        dur = dict(zip(d.iloc[:,0], d.iloc[:,1]/1000.0))
    except Exception:
        pass
    out = []
    for _, r in df.iterrows():
        out.append(Utterance(
            audio_path=str(r.get('path','')),
            transcript=str(r.get('sentence','')),
            speaker=str(r.get('client_id','')),
            dialect='MSA',                                   # Common Voice Arabic is mostly MSA
            gender=str(r.get('gender','unknown') or 'unknown')[:1] or 'unknown',
            duration_s=float(dur.get(str(r.get('path','')), 4.5)),
            corpus='CommonVoice'))
    return out

def make_synthetic(seed=7):
    rng = np.random.default_rng(seed)
    # Corpus A: broadcast MSA; Corpus B: multi-dialect web speech
    sents = ['\u0648\u0635\u0644 \u0627\u0644\u0648\u0641\u062f',  # wasala al-wafd 'the delegation arrived'
             '\u0623\u064a\u0646 \u0627\u0644\u0645\u062d\u0637\u0629',  # ayna al-mahatta 'where is the station'
             '\u0634\u0648 \u0623\u062e\u0628\u0627\u0631\u0643']        # shu akhbarak (Levantine) 'how are you'
    utts = []
    for i in range(220):                                     # Corpus A
        sp = f'A{rng.integers(1,26):03d}'
        utts.append(Utterance(f'a/{i}.wav', sents[0], sp, 'MSA',
                              rng.choice(['m','f'],p=[0.7,0.3]), float(rng.uniform(3,18)), 'BroadcastMSA'))
    for i in range(180):                                     # Corpus B
        dia = rng.choice(['Gulf','Egyptian','Levantine','Maghrebi'], p=[0.35,0.3,0.25,0.10])
        sp = f'B{rng.integers(1,41):03d}'
        utts.append(Utterance(f'b/{i}.wav', sents[2], sp, dia,
                              rng.choice(['m','f','unknown'],p=[0.5,0.4,0.1]), float(rng.uniform(2,12)), 'DialectWeb'))
    return utts

try:
    rows = load_commonvoice(CV_TSV)
    print(f'Loaded {len(rows)} real Common Voice rows.')
except Exception as e:
    rows = make_synthetic()
    print('Common Voice TSV not found; using synthetic samples instead.')

df = to_frame(rows)
print(df.shape, 'utterances')
df.head()


## 3. The statistics that drive a corpus choice

Before using a corpus you want the four numbers from Chapter 7: how many hours, how many speakers, and how those hours split across dialect and gender.


In [ ]:
def hours(x): return round(x.sum()/3600.0, 2)

print('TOTAL hours :', hours(df.duration_s))
print('TOTAL speakers:', df.speaker.nunique())
print()
print('Hours per corpus:')
print(df.groupby('corpus').duration_s.apply(hours).to_string())
print()
print('Hours per dialect:')
print(df.groupby('dialect').duration_s.apply(hours).sort_values(ascending=False).to_string())
print()
print('Hours per gender:')
print(df.groupby('gender').duration_s.apply(hours).to_string())
print()
print('Speakers per dialect:')
print(df.groupby('dialect').speaker.nunique().sort_values(ascending=False).to_string())


## 4. Leak check: speaker overlap across splits

A test set must share no speakers with training, or the score measures speaker memorisation, not recognition (Chapter 3, Chapter 7). 
This checks an existing split and shows a simple speaker-disjoint splitter.


In [ ]:
def speaker_overlap(train, test):
    s = set(train.speaker) & set(test.speaker)
    return sorted(s)

# A NAIVE random split (by row) usually leaks speakers across train/test:
shuf = df.sample(frac=1.0, random_state=0).reset_index(drop=True)
cut = int(0.8*len(shuf))
naive_train, naive_test = shuf.iloc[:cut], shuf.iloc[cut:]
leak = speaker_overlap(naive_train, naive_test)
print(f'Naive row split  -> {len(leak)} speakers appear in BOTH train and test (leak!)')

def speaker_disjoint_split(df, test_frac=0.2, seed=0):
    rng = np.random.default_rng(seed)
    spk = df.speaker.unique(); rng.shuffle(spk)
    n_test = max(1, int(test_frac*len(spk)))
    test_spk = set(spk[:n_test])
    test = df[df.speaker.isin(test_spk)]; train = df[~df.speaker.isin(test_spk)]
    return train, test

tr, te = speaker_disjoint_split(df)
print(f'Speaker-disjoint -> {len(speaker_overlap(tr, te))} overlapping speakers (good); '
      f'train {hours(tr.duration_s)} h / test {hours(te.duration_s)} h')


## 5. Inter-annotator agreement: Fleiss' kappa

When several annotators label the same items (for example the dialect of each clip), Fleiss' kappa measures how much they agree beyond chance. 
This is the computation behind Exercise 3. A value above about 0.6 is usually considered usable, though the threshold depends on the task.


In [ ]:
def fleiss_kappa(table):
    """table[i][j] = number of annotators who put item i in category j (constant raters per item)."""
    table = np.asarray(table, dtype=float)
    N, k = table.shape
    n = table.sum(axis=1)[0]                       # raters per item (assumed constant)
    p_j = table.sum(axis=0) / (N*n)                # category proportions
    P_i = (np.square(table).sum(axis=1) - n) / (n*(n-1))   # per-item agreement
    P_bar = P_i.mean()
    P_e = np.square(p_j).sum()
    return (P_bar - P_e) / (1 - P_e)

# 8 clips, 3 annotators each, 4 dialect classes [MSA, Gulf, Egyptian, Levantine]
# rows = clips; entries = how many of the 3 annotators chose each class
counts = [
    [3,0,0,0],   # all agree MSA
    [0,3,0,0],   # all agree Gulf
    [2,1,0,0],   # 2 MSA, 1 Gulf
    [0,0,3,0],   # all Egyptian
    [0,0,2,1],   # 2 Egyptian, 1 Levantine
    [0,1,0,2],   # split
    [0,0,0,3],   # all Levantine
    [1,1,1,0],   # full disagreement
]
k = fleiss_kappa(counts)
print(f"Fleiss' kappa = {k:.3f}")
print('interpretation:', 'usable (>0.6)' if k>0.6 else 'moderate' if k>0.4 else 'low; revise the guidelines')


## 6. Takeaways

- One shared schema (audio, transcript, speaker, dialect, gender, duration) lets you compare and combine corpora.
- Read a corpus by hours, speakers, and the dialect and gender breakdown before training; the dialect long tail is usually the constraint.
- Split by speaker, never by row, and verify zero speaker overlap.
- Report inter-annotator agreement (Fleiss' kappa) alongside the labels, not just the final labels.


## Exercise solutions (Exercise 3)

**Compute Fleiss' kappa for 20 clips, 3 annotators, 4 classes.** Build the per-clip count table (each row sums to 3) and pass it to `fleiss_kappa` above. 
A value above about 0.6 suggests usable agreement. The one-page data card for the resulting set must record: motivation, composition (clips, speakers, dialects), collection and consent, the annotation convention and the agreement value, the license, the splits, and known limitations.
